# 06 — GCAD Re-run with Calibrated Parameters

Update the GCAD parts library with the AL-calibrated parameters and rerun circuit mining. This discovers new topologies optimized for the real (calibrated) biological system.

**Inputs:** `workspace/calibrated_parts.pkl`
**Output:** `results/Miner_PostAL_<date>/`

In [ ]:
import os
# Navigate to repo root so all relative paths resolve correctly.
if os.path.basename(os.getcwd()) in ('notebooks', 'examples'):
    os.chdir('..')
print(f"Working directory: {os.getcwd()}")

In [ ]:
import os
import pickle
import numpy as np
import datetime
from active_learning.gcad_utils import update_gcad_library, run_gcad_miner, extract_top_circuits
from gcad.circuit import Topo  # required for pickle deserialization

## Load Calibrated Parameters

In [ ]:
with open("workspace/calibrated_parts.pkl", "rb") as f:
    calibrated_parts = pickle.load(f)

print("Calibrated parameters:")
for part, vals in calibrated_parts.items():
    print(f"  {part}: {np.round(vals, 6)}")

## Update GCAD Library and Rerun Mining

In [ ]:
# Overwrite gcad/data/parts.pkl with calibrated values (backup is created automatically)
update_gcad_library(calibrated_parts)

# Run GCAD evolution with calibrated parameters
settings_path = os.path.abspath(os.path.join("data", "gcad_results", "FreeSearch_settings.json"))
timestamp = datetime.datetime.now().strftime("%Y-%m-%d")
output_folder = os.path.join("results", f"Miner_PostAL_{timestamp}")

run_gcad_miner(settings_path, output_folder)
print(f"Results saved to: {output_folder}")

## Analyze Post-Calibration Pareto Front

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx

TOPO_COLORS = ['#e6194B', '#f58231', '#ffe119', '#bfef45', '#3cb44b',
               '#42d4f4', '#000075', '#911eb4', '#f032e6', '#000000']

def abstract_edges(edge_list):
    generic = []
    for u, v in edge_list:
        src = 'P1' if u == 'P1' else ('A' if u.startswith('Z') else ('R' if u.startswith('I') else u))
        tgt = 'Rep' if v == 'Rep' else ('A' if v.startswith('Z') else ('R' if v.startswith('I') else v))
        generic.append((src, tgt))
    return tuple(sorted(generic))

df = pd.read_pickle(os.path.join(output_folder, "final_objectives_df.pkl"))
with open(os.path.join(output_folder, "final_population.pkl"), "rb") as f:
    pop = pickle.load(f)

df['circuit_object'] = [ind[0] for ind in pop]
if 'prominence_rel' in df.columns and df['prominence_rel'].mean() < 0:
    df['prominence_rel'] = -df['prominence_rel']

df['topology_sig'] = df['circuit_object'].apply(lambda c: str(tuple(sorted(c.edge_list))))
df['abstract_sig'] = df['circuit_object'].apply(lambda c: abstract_edges(c.edge_list))
valid_df = df[df['prominence_rel'] > 0.1].copy()

topo_counts = valid_df['topology_sig'].value_counts()
sorted_sigs = topo_counts.index.tolist()
sig_to_color = {sig: TOPO_COLORS[i % len(TOPO_COLORS)] for i, sig in enumerate(sorted_sigs)}

# Pareto front scatter
history_df = pd.read_pickle(os.path.join(output_folder, "unique_objectives_df.pkl"))
if history_df['prominence_rel'].mean() < 0:
    history_df['prominence_rel'] = -history_df['prominence_rel']

plt.figure(figsize=(10, 6))
plt.scatter(history_df['t_pulse'], history_df['prominence_rel'],
            c='#757c80', s=40, alpha=0.6, linewidths=0, label='Search Space')
for sig in sorted_sigs:
    subset = valid_df[valid_df['topology_sig'] == sig]
    plt.scatter(subset['t_pulse'], subset['prominence_rel'],
                c=sig_to_color[sig], s=40, alpha=1.0, zorder=10,
                label=f"Topo (n={topo_counts[sig]})")
plt.gca().invert_xaxis()
plt.xlabel("Time to Pulse (hr) [Inverted]", fontweight='bold')
plt.ylabel("Prominence (Relative)", fontweight='bold')
plt.title("Post-Calibration Pareto Front", fontweight='bold', fontsize=14)
if len(sorted_sigs) < 15:
    plt.legend(loc='upper right')
plt.grid(True, alpha=0.2)
plt.show()

In [ ]:
# Circuit schematics (color-matched to Pareto plot)
num = len(sorted_sigs)
cols = min(num, 4)
rows = max(1, -(-num // cols))  # ceiling division

fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 4 * rows))
axes = np.array(axes).flatten()

for i, sig in enumerate(sorted_sigs):
    ax = axes[i]
    rep_circuit = valid_df[valid_df['topology_sig'] == sig].sort_values('prominence_rel', ascending=False).iloc[0]['circuit_object']
    color = sig_to_color[sig]
    G = nx.DiGraph(rep_circuit.edge_list)
    pos = nx.spring_layout(G, seed=25)
    nx.draw(G, pos, ax=ax, node_color='white', edgecolors=color, linewidths=2, node_size=700,
            with_labels=True, font_weight='bold', font_size=11, edge_color=color, width=2.5, arrowsize=15)
    ax.set_title(f"Topo {i} (n={topo_counts[sig]})", color=color, fontweight='bold', fontsize=14)
    dose_str = "\n".join([f"{k}: {v:.2f}ng" for k, v in rep_circuit.dose.items() if k != 'Rep'])
    ax.text(0.5, -0.15, dose_str, transform=ax.transAxes, ha='center', va='top', fontsize=10,
            bbox=dict(boxstyle="round,pad=0.3", edgecolor='gray', facecolor='white', alpha=0.8))

for j in range(len(sorted_sigs), len(axes)):
    axes[j].axis('off')

plt.tight_layout()
plt.show()

# Architecture summary
abs_counts = valid_df['abstract_sig'].value_counts().reset_index()
abs_counts.columns = ['abstract_sig', 'Count']
print("\nDominant architectures (post-calibration):")
for i, row in abs_counts.head(5).iterrows():
    print(f"  Arch {i+1} (n={row['Count']}): {row['abstract_sig']}")